# Final Model Full Run

This notebook uses a single prepared dataset at `artifacts/model_dataset_full.parquet`.

Experiment design:
1. Load the full prepared dataset once
2. Confirm the previously dropped perfectly correlated feature is still excluded
3. Define two feature views inside the notebook:
   - `no_als`: base features only
   - `with_als`: base features + ALS features
4. Tune CatBoost only on the `no_als` feature view
5. Run every model twice and compare whether ALS improves the results

## Imports

In [1]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## Config And Dataset

In [2]:
PROJECT_ROOT = Path('/Users/bettinabopeng/Documents/GitHub/5971')
DATASET_PATH = PROJECT_ROOT / 'artifacts' / 'model_dataset_full.parquet'
METADATA_PATH = DATASET_PATH.with_suffix('.metadata.json')

RANDOM_STATE = 42
OUTER_SPLITS = 3
INNER_SPLITS = 3
TOP_K_FIXED = 11
CATBOOST_N_ITER_SEARCH = 12

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Missing dataset: {DATASET_PATH}. Build it first with build_model_dataset.py.'
    )
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing metadata: {METADATA_PATH}. Build it first with build_model_dataset.py.'
    )

model_df = pd.read_parquet(DATASET_PATH)
metadata = json.loads(METADATA_PATH.read_text())

base_feature_cols = metadata['base_feature_cols']
als_feature_cols = metadata['als_feature_cols']
catboost_cat_cols = metadata['catboost_cat_cols']

feature_sets = {
    'no_als': base_feature_cols,
    'with_als': base_feature_cols + als_feature_cols,
}

print({
    'shape': model_df.shape,
    'users': int(model_df['user_id'].nunique()),
    'positive_rate': round(float(model_df['label'].mean()), 6),
    'als_status': metadata.get('als_status'),
})
print('Base feature count:', len(base_feature_cols))
print('ALS feature count:', len(als_feature_cols))
print('CatBoost categorical features:', catboost_cat_cols)

{'shape': (8474661, 31), 'users': 131209, 'positive_rate': 0.0978, 'als_status': 'enabled'}
Base feature count: 23
ALS feature count: 2
CatBoost categorical features: ['aisle_id', 'department_id']


## Sanity Checks

In [3]:
assert 'up_reorder_cnt' not in model_df.columns
assert 'up_reorder_cnt' not in base_feature_cols
print('Confirmed: up_reorder_cnt is not present in the final dataset.')
print('Base features after the perfect-correlation cleanup:')
print(base_feature_cols)

Confirmed: up_reorder_cnt is not present in the final dataset.
Base features after the perfect-correlation cleanup:
['up_orders_since_last', 'up_days_since_last', 'up_freq', 'up_buy_cnt', 'up_reorder_ratio', 'p_reorder_ratio', 'u_total_orders', 'cluster_product_target_rate', 'p_avg_cart_order', 'u_reorder_ratio', 'u_unique_products', 'p_total_purchases', 'up_first_order', 'u_avg_days_between_orders', 'p_unique_users', 'up_last_order', 'u_total_items', 'u_avg_basket_size', 'up_avg_cart_order', 'apriori_rule_hits', 'user_cluster_1', 'user_cluster_2', 'user_cluster_3']


## Evaluation Helpers

In [4]:
def eval_row_level(y_true, y_prob):
    y_prob_clip = np.clip(y_prob, 1e-15, 1 - 1e-15)
    return {
        'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc': average_precision_score(y_true, y_prob),
        'logloss': log_loss(y_true, y_prob_clip),
    }


def eval_order_topk(df_eval, prob_col='prob', k=11, label_col='label'):
    ranked = df_eval.sort_values(['user_id', prob_col, 'product_id'], ascending=[True, False, True]).copy()
    ranked['rank_within_user'] = ranked.groupby('user_id').cumcount() + 1
    ranked['pred_topk'] = (ranked['rank_within_user'] <= k).astype(int)

    precision_vals, recall_vals, f1_vals, hit_vals = [], [], [], []
    for _, g in ranked.groupby('user_id'):
        y_t = g[label_col].to_numpy()
        y_p = g['pred_topk'].to_numpy()
        precision_vals.append(precision_score(y_t, y_p, zero_division=0))
        recall_vals.append(recall_score(y_t, y_p, zero_division=0))
        f1_vals.append(f1_score(y_t, y_p, zero_division=0))
        hit_vals.append(float((g.loc[g['pred_topk'] == 1, label_col].sum() > 0)))

    return {
        f'precision@{k}': float(np.mean(precision_vals)),
        f'recall@{k}': float(np.mean(recall_vals)),
        f'f1@{k}': float(np.mean(f1_vals)),
        f'hit@{k}': float(np.mean(hit_vals)),
    }


def make_dynamic_k_map(df_input):
    user_k_df = df_input.groupby('user_id')['u_avg_basket_size'].first().reset_index()
    user_k_df['pred_k'] = np.rint(user_k_df['u_avg_basket_size']).astype(int).clip(lower=1)
    return dict(zip(user_k_df['user_id'], user_k_df['pred_k']))


def eval_order_dynamic_k(df_eval, prob_col='prob', k_pred_map=None, label_col='label'):
    ranked = df_eval.sort_values(['user_id', prob_col, 'product_id'], ascending=[True, False, True]).copy()
    ranked['rank_within_user'] = ranked.groupby('user_id').cumcount() + 1
    ranked['pred_k'] = ranked['user_id'].map(k_pred_map).fillna(TOP_K_FIXED).astype(int).clip(lower=1)
    ranked['pred_topk_dynamic'] = (ranked['rank_within_user'] <= ranked['pred_k']).astype(int)

    precision_vals, recall_vals, f1_vals, hit_vals, avg_k_vals = [], [], [], [], []
    for _, g in ranked.groupby('user_id'):
        y_t = g[label_col].to_numpy()
        y_p = g['pred_topk_dynamic'].to_numpy()
        precision_vals.append(precision_score(y_t, y_p, zero_division=0))
        recall_vals.append(recall_score(y_t, y_p, zero_division=0))
        f1_vals.append(f1_score(y_t, y_p, zero_division=0))
        hit_vals.append(float((g.loc[g['pred_topk_dynamic'] == 1, label_col].sum() > 0)))
        avg_k_vals.append(float(g['pred_k'].iloc[0]))

    return {
        'precision@dynamic_k': float(np.mean(precision_vals)),
        'recall@dynamic_k': float(np.mean(recall_vals)),
        'f1@dynamic_k': float(np.mean(f1_vals)),
        'hit@dynamic_k': float(np.mean(hit_vals)),
        'avg_pred_k': float(np.mean(avg_k_vals)),
    }


def summarize_results(df):
    return (
        df.groupby(['feature_set', 'model'], as_index=False)
        .agg(
            auc_mean=('auc', 'mean'),
            auc_std=('auc', 'std'),
            pr_auc_mean=('pr_auc', 'mean'),
            logloss_mean=('logloss', 'mean'),
            p11_mean=('precision@11', 'mean'),
            r11_mean=('recall@11', 'mean'),
            f11_mean=('f1@11', 'mean'),
            hit11_mean=('hit@11', 'mean'),
            pdk_mean=('precision@dynamic_k', 'mean'),
            rdk_mean=('recall@dynamic_k', 'mean'),
            fdk_mean=('f1@dynamic_k', 'mean'),
            hitdk_mean=('hit@dynamic_k', 'mean'),
            avg_pred_k=('avg_pred_k', 'mean'),
        )
        .sort_values(['model', 'feature_set'])
        .reset_index(drop=True)
    )

## Fixed Model Parameters

In [5]:
fixed_models = {
    'LogisticRegression_SAGA': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            solver='saga',
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            C=0.03,
            max_iter=150,
        )),
    ]),
    'LightGBM': lgb.LGBMClassifier(
        objective='binary',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        subsample=0.8,
        colsample_bytree=1.0,
        num_leaves=31,
        learning_rate=0.03,
        n_estimators=600,
        min_child_samples=30,
    ),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            max_iter=80,
            hidden_layer_sizes=(128, 64),
            alpha=1e-3,
            learning_rate_init=1e-3,
            batch_size=512,
            verbose=False,
        )),
    ]),
}

fixed_models

{'LogisticRegression_SAGA': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  LogisticRegression(C=0.03, class_weight='balanced',
                                     max_iter=150, n_jobs=-1, random_state=42,
                                     solver='saga'))]),
 'LightGBM': LGBMClassifier(learning_rate=0.03, min_child_samples=30, n_estimators=600,
                n_jobs=-1, objective='binary', random_state=42, subsample=0.8,
                verbosity=-1),
 'MLP': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  MLPClassifier(alpha=0.001, batch_size=512, early_stopping=True,
                                hidden_layer_sizes=(128, 64), max_iter=80,
                                n_iter_no_change=8, random_state=42))])}

## Shared Modeling Arrays

In [6]:
y = model_df['label'].astype(int).to_numpy()
groups = model_df['user_id'].to_numpy()
outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
inner_cv = GroupKFold(n_splits=INNER_SPLITS)

print('Rows:', len(model_df))
print('Unique users:', len(np.unique(groups)))

Rows: 8474661
Unique users: 131209


## CatBoost Tuning On No-ALS Only

In [7]:
catboost_base = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)

catboost_param_dist = {
    'depth': [6, 8, 10],
    'learning_rate': [0.03, 0.05, 0.08],
    'l2_leaf_reg': [3.0, 5.0, 7.0, 9.0],
    'iterations': [400, 700, 1000],
    'subsample': [0.8, 1.0],
}

In [8]:
catboost_tune_cols = feature_sets['no_als'] + catboost_cat_cols
X_catboost_tune = model_df[catboost_tune_cols].copy()
cat_feature_indices_tune = [X_catboost_tune.columns.get_loc(col) for col in catboost_cat_cols]

catboost_tuning_rows = []
for fold_id, (tr_idx, te_idx) in enumerate(outer_cv.split(X_catboost_tune, y, groups=groups), start=1):
    X_tr, X_te = X_catboost_tune.iloc[tr_idx], X_catboost_tune.iloc[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    g_tr = groups[tr_idx]
    pos_rate = float(np.mean(y_tr))

    search = RandomizedSearchCV(
        estimator=clone(catboost_base).set_params(
            scale_pos_weight=(1.0 - pos_rate) / max(pos_rate, 1e-6)
        ),
        param_distributions=catboost_param_dist,
        n_iter=CATBOOST_N_ITER_SEARCH,
        scoring='roc_auc',
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=1,
        refit=True,
        verbose=0,
    )
    search.fit(X_tr, y_tr, groups=g_tr, cat_features=cat_feature_indices_tune)

    y_prob = search.best_estimator_.predict_proba(X_te)[:, 1]
    fold_df = model_df.iloc[te_idx][['user_id', 'product_id', 'label', 'u_avg_basket_size']].copy()
    fold_df['prob'] = y_prob
    k_map = make_dynamic_k_map(fold_df)

    row = {
        'feature_set': 'no_als',
        'model': 'CatBoost_tuning',
        'outer_fold': fold_id,
        'inner_best_score_auc': float(search.best_score_),
        'best_params': str(search.best_params_),
    }
    row.update(eval_row_level(y_te, y_prob))
    row.update(eval_order_topk(fold_df, prob_col='prob', k=TOP_K_FIXED, label_col='label'))
    row.update(eval_order_dynamic_k(fold_df, prob_col='prob', k_pred_map=k_map, label_col='label'))
    catboost_tuning_rows.append(row)

    print(
        f"CatBoost tuning fold {fold_id} | auc={row['auc']:.4f} | pr_auc={row['pr_auc']:.4f} | "
        f"f1@11={row['f1@11']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
    )

catboost_tuning_df = pd.DataFrame(catboost_tuning_rows)
catboost_tuning_df

CatBoost tuning fold 1 | auc=0.8467 | pr_auc=0.4367 | f1@11=0.3526 | f1@dynamic_k=0.3841
CatBoost tuning fold 2 | auc=0.8462 | pr_auc=0.4347 | f1@11=0.3526 | f1@dynamic_k=0.3835
CatBoost tuning fold 3 | auc=0.8455 | pr_auc=0.4321 | f1@11=0.3516 | f1@dynamic_k=0.3830


,feature_set,model,outer_fold,inner_best_score_auc,best_params,auc,pr_auc,logloss,precision@11,recall@11,f1@11,hit@11,precision@dynamic_k,recall@dynamic_k,f1@dynamic_k,hit@dynamic_k,avg_pred_k
0,no_als,CatBoost_tuning,1,0.845585,"{'subsample': 0.8, 'learning_rate': 0.03, 'l2_...",0.846708,0.436690,0.485328,0.305878,0.586540,0.352559,0.885175,0.341230,0.531950,0.384056,0.851564,9.932481
1,no_als,CatBoost_tuning,2,0.845919,"{'subsample': 0.8, 'learning_rate': 0.03, 'l2_...",0.846156,0.434713,0.485675,0.305456,0.585720,0.352622,0.886181,0.340608,0.531219,0.383519,0.853599,9.943571
2,no_als,CatBoost_tuning,3,0.846279,"{'subsample': 1.0, 'learning_rate': 0.03, 'l2_...",0.845472,0.432076,0.486870,0.304582,0.585717,0.351565,0.885886,0.340067,0.531658,0.382965,0.852047,9.955438


In [9]:
catboost_best_params = ast.literal_eval(
    catboost_tuning_df.sort_values(['inner_best_score_auc', 'auc'], ascending=False).iloc[0]['best_params']
)
catboost_best_params

{'subsample': 1.0,
 'learning_rate': 0.03,
 'l2_leaf_reg': 9.0,
 'iterations': 1000,
 'depth': 8}

## Full Evaluation On Both Feature Views

In [10]:
def evaluate_fixed_estimator(feature_set_label, model_name, estimator, feature_cols):
    X = model_df[feature_cols].copy()
    splitter = GroupKFold(n_splits=OUTER_SPLITS)
    rows = []

    for fold_id, (tr_idx, te_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        fitted = clone(estimator)
        fitted.fit(X_tr, y_tr)
        y_prob = fitted.predict_proba(X_te)[:, 1]

        fold_df = model_df.iloc[te_idx][['user_id', 'product_id', 'label', 'u_avg_basket_size']].copy()
        fold_df['prob'] = y_prob
        k_map = make_dynamic_k_map(fold_df)

        row = {
            'feature_set': feature_set_label,
            'model': model_name,
            'outer_fold': fold_id,
            'inner_best_score_auc': np.nan,
            'best_params': 'fixed_from_previous_tuning',
        }
        row.update(eval_row_level(y_te, y_prob))
        row.update(eval_order_topk(fold_df, prob_col='prob', k=TOP_K_FIXED, label_col='label'))
        row.update(eval_order_dynamic_k(fold_df, prob_col='prob', k_pred_map=k_map, label_col='label'))
        rows.append(row)

        print(
            f"{feature_set_label} | {model_name} | fold {fold_id} | auc={row['auc']:.4f} | "
            f"pr_auc={row['pr_auc']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
        )

    return pd.DataFrame(rows)


def evaluate_catboost_fixed(feature_set_label, feature_cols, best_params):
    X = model_df[feature_cols + catboost_cat_cols].copy()
    cat_feature_indices = [X.columns.get_loc(col) for col in catboost_cat_cols]
    splitter = GroupKFold(n_splits=OUTER_SPLITS)
    rows = []

    for fold_id, (tr_idx, te_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        pos_rate = float(np.mean(y_tr))

        fitted = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            scale_pos_weight=(1.0 - pos_rate) / max(pos_rate, 1e-6),
            **best_params,
        )
        fitted.fit(X_tr, y_tr, cat_features=cat_feature_indices)
        y_prob = fitted.predict_proba(X_te)[:, 1]

        fold_df = model_df.iloc[te_idx][['user_id', 'product_id', 'label', 'u_avg_basket_size']].copy()
        fold_df['prob'] = y_prob
        k_map = make_dynamic_k_map(fold_df)

        row = {
            'feature_set': feature_set_label,
            'model': 'CatBoost',
            'outer_fold': fold_id,
            'inner_best_score_auc': np.nan,
            'best_params': str(best_params),
        }
        row.update(eval_row_level(y_te, y_prob))
        row.update(eval_order_topk(fold_df, prob_col='prob', k=TOP_K_FIXED, label_col='label'))
        row.update(eval_order_dynamic_k(fold_df, prob_col='prob', k_pred_map=k_map, label_col='label'))
        rows.append(row)

        print(
            f"{feature_set_label} | CatBoost | fold {fold_id} | auc={row['auc']:.4f} | "
            f"pr_auc={row['pr_auc']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
        )

    return pd.DataFrame(rows)

In [11]:
evaluation_frames = []

for feature_set_label, feature_cols in feature_sets.items():
    for model_name, estimator in fixed_models.items():
        evaluation_frames.append(
            evaluate_fixed_estimator(
                feature_set_label=feature_set_label,
                model_name=model_name,
                estimator=estimator,
                feature_cols=feature_cols,
            )
        )

    evaluation_frames.append(
        evaluate_catboost_fixed(
            feature_set_label=feature_set_label,
            feature_cols=feature_cols,
            best_params=catboost_best_params,
        )
    )

all_results_df = pd.concat(evaluation_frames, ignore_index=True)
all_results_df

no_als | LogisticRegression_SAGA | fold 1 | auc=0.8347 | pr_auc=0.4083 | f1@dynamic_k=0.3704
no_als | LogisticRegression_SAGA | fold 2 | auc=0.8340 | pr_auc=0.4065 | f1@dynamic_k=0.3700
no_als | LogisticRegression_SAGA | fold 3 | auc=0.8331 | pr_auc=0.4041 | f1@dynamic_k=0.3691
no_als | LightGBM | fold 1 | auc=0.8458 | pr_auc=0.4356 | f1@dynamic_k=0.3829
no_als | LightGBM | fold 2 | auc=0.8453 | pr_auc=0.4348 | f1@dynamic_k=0.3829
no_als | LightGBM | fold 3 | auc=0.8445 | pr_auc=0.4321 | f1@dynamic_k=0.3820
no_als | MLP | fold 1 | auc=0.8461 | pr_auc=0.4375 | f1@dynamic_k=0.3841
no_als | MLP | fold 2 | auc=0.8455 | pr_auc=0.4356 | f1@dynamic_k=0.3836
no_als | MLP | fold 3 | auc=0.8449 | pr_auc=0.4344 | f1@dynamic_k=0.3829
no_als | CatBoost | fold 1 | auc=0.8467 | pr_auc=0.4367 | f1@dynamic_k=0.3840
no_als | CatBoost | fold 2 | auc=0.8462 | pr_auc=0.4348 | f1@dynamic_k=0.3837
no_als | CatBoost | fold 3 | auc=0.8455 | pr_auc=0.4321 | f1@dynamic_k=0.3830
with_als | LogisticRegression_SAGA

,feature_set,model,outer_fold,inner_best_score_auc,best_params,auc,pr_auc,logloss,precision@11,recall@11,f1@11,hit@11,precision@dynamic_k,recall@dynamic_k,f1@dynamic_k,hit@dynamic_k,avg_pred_k
0,no_als,LogisticRegression_SAGA,1,NaN,fixed_from_previous_tuning,0.834671,0.408258,0.504208,0.296452,0.572045,0.342538,0.881859,0.328876,0.513410,0.370433,0.845459,9.932481
1,no_als,LogisticRegression_SAGA,2,NaN,fixed_from_previous_tuning,0.833963,0.406504,0.504855,0.296374,0.572749,0.343163,0.883460,0.328208,0.513691,0.369997,0.847791,9.943571
2,no_als,LogisticRegression_SAGA,3,NaN,fixed_from_previous_tuning,0.833095,0.404081,0.505902,0.295297,0.571581,0.341692,0.882548,0.327282,0.513532,0.369068,0.846057,9.955438
3,no_als,LightGBM,1,NaN,fixed_from_previous_tuning,0.845753,0.435588,0.239715,0.305371,0.585671,0.351965,0.884672,0.340332,0.529983,0.382933,0.849986,9.932481
4,no_als,LightGBM,2,NaN,fixed_from_previous_tuning,0.845347,0.434841,0.239814,0.305138,0.585478,0.352315,0.886295,0.340068,0.530432,0.382865,0.854033,9.943571
5,no_als,LightGBM,3,NaN,fixed_from_previous_tuning,0.844477,0.432100,0.240099,0.303979,0.584958,0.350892,0.886069,0.339263,0.530078,0.382018,0.851224,9.955438
6,no_als,MLP,1,NaN,fixed_from_previous_tuning,0.846051,0.437465,0.239469,0.305986,0.586792,0.352606,0.886135,0.341312,0.532135,0.384118,0.851861,9.932481
7,no_als,MLP,2,NaN,fixed_from_previous_tuning,0.845536,0.435649,0.239738,0.305612,0.585967,0.352837,0.886387,0.340618,0.531479,0.383573,0.853942,9.943571
8,no_als,MLP,3,NaN,fixed_from_previous_tuning,0.844862,0.434356,0.239879,0.304731,0.585809,0.351690,0.885635,0.340209,0.530906,0.382942,0.851087,9.955438
9,no_als,CatBoost,1,NaN,"{'subsample': 1.0, 'learning_rate': 0.03, 'l2_...",0.846705,0.436705,0.485318,0.305936,0.586666,0.352631,0.885220,0.341179,0.531981,0.384006,0.851244,9.932481


## Summaries And ALS Comparison

In [12]:
summary_df = summarize_results(all_results_df)
summary_df

,feature_set,model,auc_mean,auc_std,pr_auc_mean,logloss_mean,p11_mean,r11_mean,f11_mean,hit11_mean,pdk_mean,rdk_mean,fdk_mean,hitdk_mean,avg_pred_k
0,no_als,CatBoost,0.846121,0.000619,0.434539,0.485951,0.305366,0.586129,0.352327,0.885793,0.340664,0.531715,0.383553,0.852442,9.94383
1,with_als,CatBoost,0.846831,0.000617,0.434831,0.485259,0.305701,0.586789,0.352733,0.886052,0.340849,0.531702,0.383707,0.852320,9.94383
2,no_als,LightGBM,0.845192,0.000652,0.434176,0.239876,0.304829,0.585369,0.351724,0.885679,0.339888,0.530164,0.382606,0.851748,9.94383
3,with_als,LightGBM,0.845923,0.000628,0.434648,0.239565,0.305232,0.586129,0.352213,0.886075,0.340323,0.530999,0.383118,0.852175,9.94383
4,no_als,LogisticRegression_SAGA,0.833910,0.000789,0.406281,0.504988,0.296041,0.572125,0.342465,0.882622,0.328122,0.513545,0.369833,0.846436,9.94383
5,with_als,LogisticRegression_SAGA,0.834443,0.000786,0.406417,0.504515,0.296254,0.572355,0.342667,0.882843,0.328334,0.513726,0.370027,0.846100,9.94383
6,no_als,MLP,0.845483,0.000596,0.435824,0.239695,0.305443,0.586190,0.352378,0.886052,0.340713,0.531506,0.383544,0.852297,9.94383
7,with_als,MLP,0.846185,0.000578,0.436255,0.239461,0.305618,0.586464,0.352615,0.885854,0.340899,0.531706,0.383757,0.851771,9.94383


In [13]:
als_delta_rows = []
for model in summary_df['model'].unique():
    model_slice = summary_df[summary_df['model'] == model].set_index('feature_set')
    if {'no_als', 'with_als'}.issubset(model_slice.index):
        als_delta_rows.append({
            'model': model,
            'delta_auc_mean': model_slice.loc['with_als', 'auc_mean'] - model_slice.loc['no_als', 'auc_mean'],
            'delta_pr_auc_mean': model_slice.loc['with_als', 'pr_auc_mean'] - model_slice.loc['no_als', 'pr_auc_mean'],
            'delta_f11_mean': model_slice.loc['with_als', 'f11_mean'] - model_slice.loc['no_als', 'f11_mean'],
            'delta_fdk_mean': model_slice.loc['with_als', 'fdk_mean'] - model_slice.loc['no_als', 'fdk_mean'],
            'delta_logloss_mean': model_slice.loc['with_als', 'logloss_mean'] - model_slice.loc['no_als', 'logloss_mean'],
        })

als_delta_df = pd.DataFrame(als_delta_rows).sort_values('model').reset_index(drop=True)
als_delta_df

,model,delta_auc_mean,delta_pr_auc_mean,delta_f11_mean,delta_fdk_mean,delta_logloss_mean
0,CatBoost,0.000710,0.000293,0.000406,0.000155,-0.000693
1,LightGBM,0.000730,0.000472,0.000489,0.000513,-0.000311
2,LogisticRegression_SAGA,0.000534,0.000136,0.000202,0.000194,-0.000473
3,MLP,0.000702,0.000432,0.000237,0.000213,-0.000234


## Save Outputs

In [14]:
catboost_tuning_df.to_csv(PROJECT_ROOT / 'artifacts' / 'catboost_tuning_no_als.csv', index=False)
all_results_df.to_csv(PROJECT_ROOT / 'artifacts' / 'final_model_full_run_fold_metrics.csv', index=False)
summary_df.to_csv(PROJECT_ROOT / 'artifacts' / 'final_model_full_run_summary.csv', index=False)
als_delta_df.to_csv(PROJECT_ROOT / 'artifacts' / 'final_model_als_delta_summary.csv', index=False)

print('Saved outputs under', PROJECT_ROOT / 'artifacts')

Saved outputs under /Users/bettinabopeng/Documents/GitHub/5971/artifacts
